# HVAC Equipment Health — Notebook 02: Feature Engineering

**Goals:**
- Implement and validate COP, ΔT, load ratio, rolling stats, time features
- Document every feature with physical interpretation (key for interview)
- Validate features make physical sense: check COP range, ΔT sign, etc.
- Save processed features to `data/processed/features.parquet`

**Key deliverable:** `src/features.py` functions are tested here before being used in training.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from src.features import load_raw, build_features, get_feature_matrix, FEATURE_COLS

sns.set_theme(style='darkgrid')
figures = Path('../figures')
processed = Path('../data/processed')
processed.mkdir(exist_ok=True)
print('Setup complete')

## 1. Load Data and Build Features

In [ ]:
df_raw = load_raw('../data/raw/')
df = build_features(df_raw)
df.head()

## 2. COP Distribution

**Physical expectation:** commercial chillers typically have COP between 2.5 and 6.0 at design conditions.
- COP < 2.0 at normal outdoor temperatures = efficiency problem
- COP > 7.0 = suspect (likely a data issue or partial load measurement)

Document the actual distribution and set contamination parameter based on tails.

In [ ]:
# TODO: Plot COP distribution, annotate mean ± std, flag outlier tails
# Save to figures/02_cop_distribution.png

## 3. ΔT Features Validation

**Physical expectation:**
- ΔT supply should be positive and in range [5°C, 15°C] for normal operation
- ΔT refrigerant should be positive; narrows as refrigerant charge depletes

Check both features pass sanity bounds.

In [ ]:
# TODO: Scatter plot COP vs delta_t_refrigerant_proxy
# Expect: lower ΔT refrigerant correlates with lower COP (charge depletion signal)

## 4. Load Ratio Distribution

Load ratio > 1.0 means the building is demanding more than rated capacity.
High load ratio + declining COP = highest-risk anomaly category.

In [ ]:
# TODO: Histogram of load_ratio, annotate % of hours with load_ratio > 0.9

## 5. Rolling COP Deviation

This is the most important trend-based feature — captures slow degradation that threshold alarms miss.
A unit whose 30-day rolling COP mean is trending down by 0.1–0.2 per month is degrading before any alarm trips.

In [ ]:
# TODO: Pick 2 buildings, plot COP and cop_30d_rolling_mean together
# One healthy (stable COP), one degrading (declining rolling mean)
# Save to figures/02_cop_rolling_deviation.png

## 6. Feature Correlation Matrix

Check for multicollinearity before passing to Isolation Forest.
IF handles correlated features fine, but high correlation can help interpret SHAP values.

In [ ]:
X, feat_names = get_feature_matrix(df)
corr = X.corr()
# TODO: Heatmap, save to figures/02_feature_correlation.png

## 7. Save Processed Features

In [ ]:
df.to_parquet(processed / 'features.parquet', index=False)
print(f'Saved {len(df):,} rows to data/processed/features.parquet')
print(f'Features: {feat_names}')

## Summary

| Feature | Range observed | Notes |
|---------|---------------|-------|
| cop_proxy | TBD | Expected 2–6 |
| delta_t_supply_proxy | TBD | |
| load_ratio | TBD | % > 0.9: TBD |
| cop_deviation_from_baseline | TBD | Trend signal |